# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdelrahmanshaheen1/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np

# Read your private Hugging Face token from Colab Secrets.
# Never paste the token directly into this public notebook.
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. Open Colab Secrets and add it."
    )

# Connect DuckDB to the protected Hugging Face warehouse.
con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# Information available when the decision is made.
MARCH_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

# Future month used only for evaluation.
APRIL_DAILY = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-04/*.parquet'"
    f")"
)

print("Warehouse connection configured successfully.")
print("Feature window: March 2026")
print("Outcome window: April 2026")

Warehouse connection configured successfully.
Feature window: March 2026
Outcome window: April 2026


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
# Build the same honest March-to-April page frame used in ML-04.
# March provides the signals. April is used only to evaluate them.

feature_frame = con.sql(f"""
    WITH march_features AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(COALESCE(gsc_impressions, 0))
                AS march_impressions,

            SUM(COALESCE(gsc_clicks, 0))
                AS march_clicks,

            ROUND(
                100.0 * SUM(COALESCE(gsc_clicks, 0))
                / NULLIF(SUM(COALESCE(gsc_impressions, 0)), 0),
                4
            ) AS march_ctr_pct,

            AVG(
                CASE
                    WHEN gsc_impressions > 0
                         AND gsc_avg_position > 0
                    THEN gsc_avg_position
                END
            ) AS march_avg_position,

            COUNT(
                DISTINCT CASE
                    WHEN gsc_impressions > 0
                    THEN report_date
                END
            ) AS march_active_days

        FROM {MARCH_DAILY}

        GROUP BY
            client_hash_id,
            content_hash_id

        HAVING SUM(COALESCE(gsc_impressions, 0)) >= 100
    ),

    april_outcomes AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(COALESCE(gsc_impressions, 0))
                AS april_impressions,

            COUNT(DISTINCT report_date)
                AS april_observed_days

        FROM {APRIL_DAILY}

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        m.*,
        a.april_impressions,

        CASE
            WHEN a.april_impressions
                 < 0.80 * m.march_impressions
            THEN 1
            ELSE 0
        END AS is_next_month_decline

    FROM march_features AS m

    INNER JOIN april_outcomes AS a
        USING (client_hash_id, content_hash_id)

    WHERE a.april_observed_days > 0
""").df()

print("Final unit: one row per client and webpage.")
print(f"Rows available for signal checks: {len(feature_frame):,}")
print(
    "Overall April-decline rate: "
    f"{feature_frame['is_next_month_decline'].mean():.1%}"
)


# ============================================================
# SIGNAL 1 — CTR compared with similar search positions
# ============================================================

signal_frame = feature_frame.copy()

signal_frame["position_bucket"] = pd.cut(
    signal_frame["march_avg_position"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=[
        "top_3",
        "positions_4_10",
        "positions_11_20",
        "positions_21_50",
        "positions_51_plus",
    ],
    include_lowest=True,
)

# Compare each page's CTR with the median CTR of pages
# in the same position bucket.
signal_frame["position_bucket_median_ctr"] = (
    signal_frame
    .groupby("position_bucket", observed=True)["march_ctr_pct"]
    .transform("median")
)

signal_frame["ctr_signal"] = np.where(
    signal_frame["march_ctr_pct"]
    < signal_frame["position_bucket_median_ctr"],
    "below_position_median",
    "at_or_above_position_median",
)

ctr_bucket_table = (
    signal_frame
    .dropna(
        subset=[
            "position_bucket",
            "march_ctr_pct",
            "position_bucket_median_ctr",
        ]
    )
    .groupby(
        ["position_bucket", "ctr_signal"],
        observed=True,
    )
    .agg(
        n=("is_next_month_decline", "size"),
        decline_rate=("is_next_month_decline", "mean"),
        mean_ctr_pct=("march_ctr_pct", "mean"),
    )
    .reset_index()
)

ctr_bucket_table["decline_rate"] = (
    100 * ctr_bucket_table["decline_rate"]
).round(1)

ctr_bucket_table["mean_ctr_pct"] = (
    ctr_bucket_table["mean_ctr_pct"]
).round(3)

print("\nSIGNAL 1 — CTR compared with similar positions")
display(ctr_bucket_table)


# ============================================================
# SIGNAL 2 — March impressions / visibility
# ============================================================

signal_frame["volume_bucket"] = pd.qcut(
    signal_frame["march_impressions"],
    q=4,
    labels=[
        "Q1_lowest_visibility",
        "Q2",
        "Q3",
        "Q4_highest_visibility",
    ],
)

volume_bucket_table = (
    signal_frame
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("is_next_month_decline", "size"),
        minimum_impressions=("march_impressions", "min"),
        maximum_impressions=("march_impressions", "max"),
        decline_rate=("is_next_month_decline", "mean"),
    )
    .reset_index()
)

volume_bucket_table["decline_rate"] = (
    100 * volume_bucket_table["decline_rate"]
).round(1)

print("\nSIGNAL 2 — March visibility")
display(volume_bucket_table)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final unit: one row per client and webpage.
Rows available for signal checks: 101,441
Overall April-decline rate: 51.7%

SIGNAL 1 — CTR compared with similar positions


,position_bucket,ctr_signal,n,decline_rate,mean_ctr_pct
0,top_3,at_or_above_position_median,4149,39.1,0.651
1,top_3,below_position_median,4146,67.7,0.099
2,positions_4_10,at_or_above_position_median,23266,39.8,0.596
3,positions_4_10,below_position_median,23265,61.5,0.054
4,positions_11_20,at_or_above_position_median,10870,46.4,0.472
5,positions_11_20,below_position_median,10868,60.0,0.009
6,positions_21_50,at_or_above_position_median,20928,51.4,0.139
7,positions_51_plus,at_or_above_position_median,3949,54.8,0.045



SIGNAL 2 — March visibility


,volume_bucket,n,minimum_impressions,maximum_impressions,decline_rate
0,Q1_lowest_visibility,25370,100.0,283.0,54.2
1,Q2,25351,284.0,786.0,54.1
2,Q3,25360,787.0,2514.0,52.1
3,Q4_highest_visibility,25360,2515.0,617124.0,46.6


### Signal verdicts and final baseline rule

**Signal 1 — CTR compared with similar positions: CONFIRMED**

For pages ranking in the top 20, pages with March CTR below the median CTR of their position bucket had a higher April decline rate than pages at or above the median. The difference was visible in the top-3, positions 4–10, and positions 11–20 buckets. Therefore, weak CTR relative to similar-position pages is retained in the baseline rule.

**Signal 2 — March visibility as a decline-risk signal: OPPOSITE**

The decline rate decreased from 54.2% in the lowest-visibility quartile to 46.6% in the highest-visibility quartile. Therefore, high impressions do not appear to indicate a higher probability of next-month decline, and visibility will not be used as a decline-risk component in this rule.

### Plain-language rule

Consider only pages with an average March position of 20 or better. If a page's March CTR is below the median CTR of pages in the same position bucket, give it a score based on how far its CTR falls below that median. A larger CTR gap receives a higher review priority.

- **Reason code:** `LOW_CTR_FOR_POSITION`
- **Action label:** `REVIEW_CTR_AND_REFRESH`
- **Score:** CTR shortfall from the position-bucket median, scaled from 0 to 100

The rule uses only March information. April measurements and the future decline label are used only to evaluate the queue.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import os

ranked_queue = signal_frame.copy()

supported_buckets = [
    "top_3",
    "positions_4_10",
    "positions_11_20",
]

# Calculate how far each page's CTR is below the median CTR
# of webpages in the same position bucket.
ranked_queue["ctr_gap_ratio"] = np.where(
    ranked_queue["position_bucket"].isin(supported_buckets)
    & ranked_queue["position_bucket_median_ctr"].gt(0),

    (
        (
            ranked_queue["position_bucket_median_ctr"]
            - ranked_queue["march_ctr_pct"]
        )
        / ranked_queue["position_bucket_median_ctr"]
    ).clip(lower=0, upper=1),

    0.0,
)

# The rule triggers only when:
# 1. the page ranks within the supported top-20 buckets, and
# 2. its CTR is below the median for its position bucket.
ranked_queue["rule_triggered"] = (
    ranked_queue["position_bucket"].isin(supported_buckets)
    & (
        ranked_queue["march_ctr_pct"]
        < ranked_queue["position_bucket_median_ctr"]
    )
)

# Transparent score from 0 to 100.
ranked_queue["baseline_score"] = np.where(
    ranked_queue["rule_triggered"],
    100 * ranked_queue["ctr_gap_ratio"],
    0.0,
).round(2)

ranked_queue["reason_code"] = np.where(
    ranked_queue["rule_triggered"],
    "LOW_CTR_FOR_POSITION",
    "NO_BASELINE_FLAG",
)

ranked_queue["action_label"] = np.where(
    ranked_queue["rule_triggered"],
    "REVIEW_CTR_AND_REFRESH",
    "MONITOR",
)

# Rank from highest to lowest baseline score.
ranked_queue = (
    ranked_queue
    .sort_values(
        by=["baseline_score"],
        ascending=False,
    )
    .reset_index(drop=True)
)

ranked_queue.insert(
    0,
    "baseline_rank",
    np.arange(1, len(ranked_queue) + 1),
)

# Evaluate the ranked queue using the future April outcome.
# The future label was not used to calculate the score.
precision_at_50 = (
    ranked_queue
    .head(50)["is_next_month_decline"]
    .mean()
)

triggered_count = int(
    ranked_queue["rule_triggered"].sum()
)

print(f"Pages flagged by the rule: {triggered_count:,}")
print(f"Baseline Precision@50: {precision_at_50:.3f}")

# Write the action queue without April or label-derived fields.
output_columns = [
    "baseline_rank",
    "client_hash_id",
    "content_hash_id",
    "march_impressions",
    "march_clicks",
    "march_ctr_pct",
    "march_avg_position",
    "march_active_days",
    "position_bucket",
    "position_bucket_median_ctr",
    "baseline_score",
    "reason_code",
    "action_label",
]

os.makedirs(
    "work/outputs",
    exist_ok=True,
)

output_path = (
    "work/outputs/"
    "baseline_action_score.csv"
)

ranked_queue[output_columns].to_csv(
    output_path,
    index=False,
)

print(f"Ranked queue written to: {output_path}")

ranked_queue[
    output_columns
].head(20)

Pages flagged by the rule: 38,279
Baseline Precision@50: 0.640
Ranked queue written to: work/outputs/baseline_action_score.csv


,baseline_rank,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_active_days,position_bucket,position_bucket_median_ctr,baseline_score,reason_code,action_label
0,1,client_fef1a8f436438636,content_ca72b8d0b1146e8b,243.0,0.0,0.0,11.003036,31,positions_11_20,0.0966,100.0,LOW_CTR_FOR_POSITION,REVIEW_CTR_AND_REFRESH
1,2,client_fef1a8f436438636,content_96b74e086871e5fd,559.0,0.0,0.0,5.498478,31,positions_4_10,0.1955,100.0,LOW_CTR_FOR_POSITION,REVIEW_CTR_AND_REFRESH
2,3,client_fef1a8f436438636,content_8aa4a88518511bd5,402.0,0.0,0.0,4.672348,31,positions_4_10,0.1955,100.0,LOW_CTR_FOR_POSITION,REVIEW_CTR_AND_REFRESH
3,4,client_fef1a8f436438636,content_f002744131e3a456,218.0,0.0,0.0,5.269984,31,positions_4_10,0.1955,100.0,LOW_CTR_FOR_POSITION,REVIEW_CTR_AND_REFRESH
4,5,client_fef1a8f436438636,content_30f075d808c989f8,439.0,0.0,0.0,5.156590,31,positions_4_10,0.1955,100.0,LOW_CTR_FOR_POSITION,REVIEW_CTR_AND_REFRESH
5,6,client_fef1a8f436438636,content_3619f667db4b48ff,347.0,0.0,0.0,2.553118,31,top_3,0.2525,100.0,LOW_CTR_FOR_POSITION,REVIEW_CTR_AND_REFRESH
6,7,client_fef1a8f436438636,content_90f8923a7e9ca7c7,564.0,0.0,0.0,3.642067,31,positions_4_10,0.1955,100.0,LOW_CTR_FOR_POSITION,REVIEW_CTR_AND_REFRESH
7,8,client_fef1a8f436438636,content_0d3992062ced2baa,396.0,0.0,0.0,4.240578,31,positions_4_10,0.1955,100.0,LOW_CTR_FOR_POSITION,REVIEW_CTR_AND_REFRESH
8,9,client_fef1a8f436438636,content_1543a3fedee9fe63,1497.0,0.0,0.0,19.938941,31,positions_11_20,0.0966,100.0,LOW_CTR_FOR_POSITION,REVIEW_CTR_AND_REFRESH
9,10,client_fef1a8f436438636,content_e7eb3f5d54079256,770.0,0.0,0.0,7.387231,31,positions_4_10,0.1955,100.0,LOW_CTR_FOR_POSITION,REVIEW_CTR_AND_REFRESH


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
top20_review = ranked_queue.head(20).copy()

def confidence_note(row):
    if row["march_active_days"] >= 28 and row["march_impressions"] >= 500:
        return "Higher confidence: strong March coverage and meaningful impressions."
    elif row["march_active_days"] >= 20:
        return "Medium confidence: reasonable March coverage, but impact may be limited."
    else:
        return "Lower confidence: limited active-day coverage may make the signal unstable."

def wrong_reason(row):
    if row["march_clicks"] == 0:
        return (
            "Could be wrong if zero clicks reflect tracking issues, unusual search intent, "
            "or a snippet problem rather than content that needs refreshing."
        )
    return (
        "Could be wrong if the CTR difference is normal for this query mix, "
        "or if the page is healthy despite being below its bucket median."
    )

top20_review["review_line"] = top20_review.apply(
    lambda row: (
        f"Rank {int(row['baseline_rank'])}: "
        f"Action={row['action_label']}; "
        f"Why={row['reason_code']} — CTR {row['march_ctr_pct']:.3f}% "
        f"versus bucket median {row['position_bucket_median_ctr']:.3f}%, "
        f"position {row['march_avg_position']:.1f}, "
        f"{int(row['march_impressions']):,} impressions. "
        f"{confidence_note(row)} "
        f"{wrong_reason(row)}"
    ),
    axis=1,
)

for line in top20_review["review_line"]:
    print(line)
    print()

Rank 1: Action=REVIEW_CTR_AND_REFRESH; Why=LOW_CTR_FOR_POSITION — CTR 0.000% versus bucket median 0.097%, position 11.0, 243 impressions. Medium confidence: reasonable March coverage, but impact may be limited. Could be wrong if zero clicks reflect tracking issues, unusual search intent, or a snippet problem rather than content that needs refreshing.

Rank 2: Action=REVIEW_CTR_AND_REFRESH; Why=LOW_CTR_FOR_POSITION — CTR 0.000% versus bucket median 0.196%, position 5.5, 559 impressions. Higher confidence: strong March coverage and meaningful impressions. Could be wrong if zero clicks reflect tracking issues, unusual search intent, or a snippet problem rather than content that needs refreshing.

Rank 3: Action=REVIEW_CTR_AND_REFRESH; Why=LOW_CTR_FOR_POSITION — CTR 0.000% versus bucket median 0.196%, position 4.7, 402 impressions. Medium confidence: reasonable March coverage, but impact may be limited. Could be wrong if zero clicks reflect tracking issues, unusual search intent, or a snip

### Top-20 review conclusion

All top-ranked pages were selected because their March CTR was far below the median CTR for pages in similar search positions. Many had zero recorded clicks, which made their CTR gap reach the maximum score.

The recommendations are plausible because the pages still had measurable impressions and ranked within the top 20. However, the review also exposes weaknesses in the baseline. Many pages receive identical scores, one client can occupy several top positions, and zero clicks may sometimes reflect tracking, query-intent, or snippet issues rather than content that needs refreshing.

Therefore, this queue should be used to prioritize human investigation, not to trigger automatic content changes.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and leakage check

Some recommendations appear weaker when they have few March impressions, limited active-day coverage, or identical scores caused by zero clicks. The queue may also overrepresent clients that have many qualifying pages. These cases show that a later model or improved baseline may need stronger evidence requirements, better tie-breaking, or client-level diversity controls.

The baseline score uses only March information:

- March CTR
- March average position
- the median March CTR for the same position bucket

April impressions and `is_next_month_decline` were used only after ranking to evaluate Precision@50. They were not used to calculate the score, reason code, or action label. No future-window field, product flag, target-derived column, URL, query, or private client information entered the rule.

The baseline result retained for comparison with the Week-5 model is **Precision@50 = 0.640**.

## Self-check

Before you submit, confirm each line honestly:

- [ ✔] Every section above is filled — markdown thinking AND the code that backs it
- [ ✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ✔] No client names, URLs, or private queries anywhere
- [ ✔] My claims use careful words: observed, measured, directional, decision-support
- [ ✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.